# Dataset 作成
- 概要
    - データ概要
- データ内容
    - shema  
    

# Spark delta table 作成

- ２つテーブル
    - ファクト表　1000万行
        - 全口座の取引明細　金額・種別（入出金、振替）・チャネル・タイムスタンプ

- ディメンション　5000口座〜2万口座
    - 口座の属性　開設日、居住地域、口座種別　リスク区分

In [0]:
#  ライブラリインポート
import random
from datetime import date, timedelta

from pyspark.sql.functions import *
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    IntegerType,
    LongType,
    TimestampType,
)

In [0]:
# パラメータ
SCHEMA = "workspace.datasets"
NUM_ACCOUNTS = 10_000
NUM_TRANSACTIONS = 10_000_000

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

## ディメンション表: account（口座属性）

In [0]:
account_schema = StructType([
    StructField("account_id", StringType(), False),
    StructField("open_date", DateType(), False),
    StructField("region", StringType(), False),
    StructField("account_type", StringType(), False),
    StructField("risk_category", StringType(), False),
])

regions = ["北海道", "東北", "関東", "中部", "近畿", "中国", "四国", "九州・沖縄"]
account_types = ["普通", "当座", "定期"]
risk_categories = ["低", "中", "高"]
risk_weights = [0.7, 0.25, 0.05]

today = date.today()

accounts = [
    {
        "account_id": f"ACC{idx + 1:05}",
        "open_date": today - timedelta(days=random.randint(0, 365 * 10)),
        "region": random.choice(regions),
        "account_type": random.choice(account_types),
        "risk_category": random.choices(risk_categories, weights=risk_weights)[0],
    }
    for idx in range(NUM_ACCOUNTS)
]

df_account = spark.createDataFrame(accounts, schema=account_schema)
df_account.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.account")

display(df_account)

In [0]:
# 作成したaccountテーブルの確認
df_account_check = spark.read.table(f"{SCHEMA}.account")
display(df_account_check.limit(5))
print(df_account_check.count())

## ファクト表: transaction（取引明細）

In [0]:
CHANNELS = ["ATM", "窓口", "ネットバンキング", "モバイルアプリ"]

df_trans = (
    spark.range(0, NUM_TRANSACTIONS)
    .withColumnRenamed("id", "transaction_id")
    .withColumn(
        "account_id",
        concat(lit("ACC"), lpad((floor(rand() * NUM_ACCOUNTS) + 1).cast("string"), 5, "0")),
    )
    .withColumn("_type_rand", rand())
    .withColumn(
        "transaction_type",
        when(col("_type_rand") < 0.45, lit("入金"))
        .when(col("_type_rand") < 0.85, lit("出金"))
        .otherwise(lit("振替")),
    )
    .withColumn(
        "counter_account_id",
        when(
            col("transaction_type") == "振替",
            concat(lit("ACC"), lpad((floor(rand() * NUM_ACCOUNTS) + 1).cast("string"), 5, "0")),
        ).otherwise(lit(None).cast(StringType())),
    )
    .withColumn(
        "channel",
        element_at(
            array(*[lit(c) for c in CHANNELS]),
            (floor(rand() * len(CHANNELS)) + 1).cast("int"),
        ),
    )
    .withColumn("amount", (rand() * 999_900 + 100).cast(IntegerType()))
    .withColumn(
        "transaction_timestamp",
        expr("timestamp_seconds(unix_timestamp(current_timestamp()) - cast(rand() * 31536000 as bigint))"),
    )
    .drop("_type_rand")
    .select(
        "transaction_id",
        "account_id",
        "counter_account_id",
        "transaction_type",
        "channel",
        "amount",
        "transaction_timestamp",
    )
)

df_trans.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.transaction")

display(df_trans.limit(5))

In [0]:
# 作成したtransactionテーブルの確認
df_trans_check = spark.read.table(f"{SCHEMA}.transaction")
display(df_trans_check.limit(5))
print(df_trans_check.count())
display(df_trans_check.groupBy("transaction_type").count())